# OTT Movies & Series - Análisis
## Predicción de calificación + Sistema de recomendación content-based

Notebook enfocado en la aplicación del flujo completo de análisis de datos en 10 fases sobre un dataset sintético de películas y series de plataformas OTT (Netflix, Prime Video, Hotstar) inspirado en IMDb.

**Variable objetivo principal:** `rating` — calificación tipo IMDb (rango aprox. 1.0 a 10.0). Problema de **regresión**.

**Por qué regresión y no clasificación:**
El dataset documenta sus casos de uso como "Ranking & Popularity Prediction" y "Recommendation Systems". El `rating` es una variable continua con magnitud real (un título de 9.2 es objetivamente mejor que uno de 7.0), por lo que tratarlo como continuo retiene más información que binarizarlo. La clasificación binaria forzaría a perder señal y ocultaría errores grandes (predecir 9.0 vs 7.5 cuenta lo mismo que predecir 9.0 vs 4.0 si ambos están en la clase "alto").

**Bonus al final (Fase 10.5):** Sistema de recomendación **content-based** con TF-IDF + similitud coseno sobre `combined_features`, demostrando un caso de uso NLP no supervisado sobre el mismo dataset.

**Problemas que evitamos:**
- Usar columnas derivadas de `rating` (como `weighted_rating`, `engagement_score`, `popularity_score`, `trending_score`) que generan **fuga de datos**.
- Pasar campos de alta cardinalidad (`content_id`, `title`) o textos largos directamente al modelo de regresión.
- Mezclar pasado y futuro: el dataset tiene componente temporal (`release_year`), por lo que la división se hace cronológicamente con `shuffle=False`.

**Enfoque aplicado:**
Construimos variables derivadas a partir de la fecha y los votos (años desde lanzamiento, votos por año, escala log) que capturan la "intensidad" de un título sin filtrar información del rating. El modelo aprende a estimar la calificación esperada a partir de **género, plataforma, país, idioma, duración y popularidad temporal**. Al final, un sistema de recomendación TF-IDF demuestra cómo encontrar títulos similares por descripción.



## Fase 0: Descarga del dataset
La data se encuentra disponible en `./data/ott_movies_series/ott_movies_clean_unique.csv`. Si necesitas re-descargarla desde Kaggle, descomenta el bloque de `kagglehub`.


In [ ]:
# import kagglehub
# import shutil
# import os
#
# path = kagglehub.dataset_download("usuario/ott-movies-series-dataset")
# print("Ruta original de descarga:", path)
#
# destino = "./data/ott_movies_series"
# shutil.copytree(path, destino, dirs_exist_ok=True)
# print("Archivos disponibles para analizar:")
# print(os.listdir(destino))

import os
print("Archivos disponibles:")
print(os.listdir('./data/ott_movies_series'))



## Fase 1: Ingesta y Exploración Inicial
Cargar los datos y entender su forma antes de modificar nada.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

# Cargar el dataset
data = pd.read_csv('./data/ott_movies_series/ott_movies_clean_unique.csv')

data.shape


In [ ]:
data.info()


In [ ]:
data.describe(include='all').T


In [ ]:
data.head()



## Fase 2: Limpieza Básica del Dataset

Aplicamos los 4 pilares: Target, Data Leakage, Cardinalidad y Redundancia.

- **Variable Objetivo (Target):**
  `rating` (calificación continua tipo IMDb). Problema de **regresión**.

- **Fuga de Datos (Data Leakage):**
  - `weighted_rating`: directamente derivada de `rating` con shrinkage.
  - `engagement_score`, `popularity_score`, `trending_score`: métricas agregadas que la documentación describe explícitamente como *"based on rating and votes"*. Si las dejamos, el modelo aprende a invertir esas fórmulas en lugar de estimar la calificación a partir del contenido. Eliminación obligatoria.

- **Alta Cardinalidad sin Valor Predictivo:**
  - `content_id`: 1 valor único por fila.
  - `title`: texto único por título.

- **Redundancia / Texto:**
  - `tags`, `combined_features`, `description`: texto que ya está representado por `type`, `genre`, `platform`, `language`. Los **conservamos en un DataFrame separado** porque los necesitamos para el sistema de recomendación de la Fase 10.5, pero los excluimos del modelo de regresión.
  - `poster_url`: URL irrelevante para el modelo.


In [ ]:
# 1. Reservar las columnas de texto para la Fase 10.5 (recomendacion NLP)
text_data = data[['content_id', 'title', 'type', 'genre', 'platform',
                  'tags', 'combined_features', 'description']].copy()

# 2. Eliminar columnas segun los 4 pilares (para el modelo de regresion)
columns_to_delete = [
    # Alta cardinalidad
    'content_id', 'title',
    # Texto / redundancia (se reservaron para NLP)
    'tags', 'combined_features', 'description', 'poster_url',
    # Fuga de datos (derivadas de rating / votes)
    'weighted_rating', 'engagement_score', 'popularity_score', 'trending_score',
]

df = data.drop(columns=columns_to_delete)

# 3. Eliminar filas con nulos criticos en features y target
columnas_criticas = ['type', 'genre', 'platform', 'country', 'language',
                     'release_year', 'duration_minutes', 'votes', 'rating']
df = df.dropna(subset=columnas_criticas)

print(f"- Dataset tras limpieza: {df.shape}")
print(f"- Columnas eliminadas: {len(columns_to_delete)}")
print(f"- Filas eliminadas por nulos criticos: {len(data) - len(df)}")
print(f"- DataFrame de texto reservado para NLP: {text_data.shape}")



### Transformación de la Columna Temporal
El dataset no trae timestamps completos, pero `release_year` es la coordenada temporal natural. Lo usamos para ordenar cronológicamente y derivar variables que capturan antigüedad.


In [ ]:
# 1. Ordenar cronologicamente (paso critico para el split temporal)
df = df.sort_values('release_year').reset_index(drop=True)

# 2. Extraer / derivar componentes utiles
ANO_REFERENCIA = 2025
df['years_since_release'] = (ANO_REFERENCIA - df['release_year']).clip(lower=1)

print(f"Rango temporal: {df['release_year'].min()} - {df['release_year'].max()}")
print(f"Anios presentes: {sorted(df['release_year'].unique())}")


In [ ]:
df.head()


## Fase 3: Análisis Exploratorio Profundo (Deep EDA)
Auditoría sistemática + gráficos para descubrir anomalías y patrones, con foco en la distribución del target `rating`.


In [ ]:
# Chequeo 1: Rangos sospechosos
df[['release_year', 'duration_minutes', 'votes', 'rating']].describe().T


In [ ]:
# Chequeo 2: Valores unicos en columnas clave
for col in ['type', 'genre', 'platform', 'country', 'language']:
    print(f"\n--- {col} ---")
    print(df[col].value_counts().head(10))


In [ ]:
# Chequeo 3: Nulos residuales
nulos = df.isnull().sum()
nulos = nulos[nulos > 0]
if len(nulos) > 0:
    print(nulos)
else:
    print("Sin nulos detectados.")


In [ ]:
# Chequeo 4: Duplicados
print(f"Filas duplicadas exactas: {df.duplicated().sum()}")


In [ ]:
# Chequeo 5: Distribucion del target (regresion)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['rating'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(df['rating'].mean(), color='red', linestyle='--', label=f"Media: {df['rating'].mean():.2f}")
axes[0].axvline(df['rating'].median(), color='orange', linestyle='--', label=f"Mediana: {df['rating'].median():.2f}")
axes[0].set_title('Distribucion del Target (rating)')
axes[0].set_xlabel('rating')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

sns.boxplot(y=df['rating'], ax=axes[1], color='lightcoral')
axes[1].set_title('Boxplot de rating')

plt.tight_layout()
plt.show()

print(f"Skewness del target: {df['rating'].skew():.3f}")
print(f"Si |skew| > 1, considerar transformar el target con log(rating).")


In [ ]:
# Boxplots de variables numericas (deteccion de outliers)
vars_plot = ['release_year', 'duration_minutes', 'votes', 'years_since_release']

fig, axes = plt.subplots(1, len(vars_plot), figsize=(16, 4))
for i, col in enumerate(vars_plot):
    sns.boxplot(y=df[col], ax=axes[i], color="skyblue")
    axes[i].set_title(f'Distribucion de {col}')
plt.tight_layout()
plt.suptitle('Boxplots - Deteccion de Outliers', y=1.02, fontsize=14)
plt.show()


In [ ]:
# Heatmap de correlacion (variables numericas vs target)
plt.figure(figsize=(8, 6))
cols_corr = ['rating', 'release_year', 'duration_minutes', 'votes', 'years_since_release']
sns.heatmap(df[cols_corr].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title('Mapa de Correlacion (incluye target)')
plt.show()

print("\nLectura: la columna 'rating' nos dice que tan ligado esta cada feature al target.")
print("Valores cercanos a 0 indican que el feature por si solo no explica la calificacion;")
print("habra que apoyarse en interacciones (encoded categoricas + numericas).")


## Fase 4: Limpieza Post-EDA
Corregir los problemas encontrados en la exploración.


In [ ]:
# Revisar outliers extremos en duration_minutes y votes.
# La duracion puede tener valores muy altos legitimamente (series largas) -> conservar.
# Los votos siguen una distribucion sesgada -> aplicaremos transformacion log en Feature Engineering.
print(f"Duracion: min={df['duration_minutes'].min()}, max={df['duration_minutes'].max()}, p99={df['duration_minutes'].quantile(0.99):.1f}")
print(f"Votos:    min={df['votes'].min()}, max={df['votes'].max()}, p99={df['votes'].quantile(0.99):.1f}")
print(f"Rating:   min={df['rating'].min()}, max={df['rating'].max()}, p99={df['rating'].quantile(0.99):.1f}")


In [ ]:
# Limpieza de nulos residuales (por seguridad)
nulos_residuales = df.isnull().sum().sum()
print(f"Nulos residuales: {nulos_residuales}")
df = df.dropna()


In [ ]:
print(f"Rango temporal limpio: {df['release_year'].min()} - {df['release_year'].max()}")
print(f"Total de filas listas para Feature Engineering: {len(df)}")


---
### Feature Engineering (LA CLAVE)

**Descubrimiento importante:** En este dataset los identificadores no aportan, pero las **categorías** se repiten lo suficiente para que el modelo encuentre patrones por género/plataforma/país/idioma. El reto está en `votes`: un título de 2015 acumula más votos que uno de 2024 sin necesariamente ser mejor, así que hay un sesgo de antigüedad que hay que descontar.

**Que vamos a crear:**
- `votes_log`: logaritmo de los votos para reducir el sesgo de la cola larga.
- `votes_per_year`: votos divididos entre años desde lanzamiento (popularidad anualizada, descontando antigüedad).
- `duration_log`: logaritmo de la duración para Series con miles de minutos.
- `is_long_content`: indicador binario de contenido largo.

**Regla anti-leakage:** ninguna variable usa `rating` ni los scores derivados. Todas se construyen a partir de columnas observables **antes** de que el público califique el título.


In [ ]:
# 1. votes_log: votos en escala logaritmica
df['votes_log'] = np.log1p(df['votes'])

# 2. votes_per_year: popularidad anualizada
df['votes_per_year'] = df['votes'] / df['years_since_release']

# 3. duration_log: duracion en escala logaritmica
df['duration_log'] = np.log1p(df['duration_minutes'])

# 4. is_long_content: indicador binario de contenido largo
mediana_duracion = df['duration_minutes'].median()
df['is_long_content'] = (df['duration_minutes'] > mediana_duracion).astype(int)

print(f"Mediana de duracion: {mediana_duracion}")
print(f"Filas con contenido largo: {df['is_long_content'].sum()} ({df['is_long_content'].mean()*100:.1f}%)")


In [ ]:
# Verificar las nuevas variables
nuevas = ['votes_log', 'votes_per_year', 'duration_log', 'is_long_content']
df[nuevas].describe().T


In [ ]:
# Eliminar filas con nulos generados por las transformaciones
df_modelo = df.dropna(subset=['votes_log', 'votes_per_year', 'duration_log']).copy()

print(f"Filas listas para modelo: {len(df_modelo)} de {len(df)} ({len(df_modelo)/len(df)*100:.1f}%)")


In [ ]:
# Validar la relacion entre votes_per_year y el target (rating promedio por bin)
bins = pd.qcut(df_modelo['votes_per_year'], q=5, duplicates='drop')
print("\nRating promedio segun votes_per_year (validacion):")
print(df_modelo.groupby(bins, observed=True)['rating'].agg(['mean', 'std', 'count']).round(3))


#### Interpretación

- Si `votes_per_year` muestra una relación monótona con `rating` (a más votos por año, mayor calificación promedio), tenemos una **feature con poder predictivo real**.
- Si la relación es plana, el contenido bien valorado no es necesariamente el más visto, y el modelo deberá apoyarse en la combinación de género/plataforma/país/idioma.
- En cualquier caso, esta variable ya descuenta el sesgo de antigüedad, por lo que es más limpia que usar `votes` crudo.
- **Conclusión:** `votes_per_year`, `duration_log` y los encodings categóricos son las features más prometedoras para el modelo de regresión.


## Fase 5: División de Datos (Train/Test Split)

- **CRÍTICO:**
  `shuffle=False` porque ya ordenamos por `release_year`. Se entrena con contenido más antiguo y se evalúa con el más reciente, simulando la situación real de producción.

- **Variables que eliminamos del modelo (y por qué):**
  - `votes` crudo: representado por `votes_log` y `votes_per_year`.
  - `duration_minutes` crudo: capturado por `duration_log` e `is_long_content`.
  - `release_year` crudo: capturado por `years_since_release`.


In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = [
    'type',
    'genre',
    'platform',
    'country',
    'language',
    'years_since_release',
    'duration_log',
    'is_long_content',
    'votes_log',
    'votes_per_year',
]

X = df_modelo[feature_cols]
y = df_modelo['rating']  # Target continuo

# Division temporal: 80% pasado (train), 20% futuro (test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

print(f"Datos de entrenamiento (pasado): {X_train.shape}")
print(f"Datos de prueba (futuro):        {X_test.shape}")
print(f"Rating - rango train: [{y_train.min():.2f}, {y_train.max():.2f}], media={y_train.mean():.2f}")
print(f"Rating - rango test:  [{y_test.min():.2f}, {y_test.max():.2f}], media={y_test.mean():.2f}")



## Fase 6: Encoding y Transformación de Variables

**Nota:** Todas nuestras variables categóricas (`type`, `genre`, `platform`, `country`, `language`) tienen cardinalidad baja (entre 2 y 7 valores). Aplicamos **One-Hot Encoding** para todas. No hace falta Target Encoding porque no hay alta cardinalidad después de la limpieza.


In [ ]:
cat_cols = ['type', 'genre', 'platform', 'country', 'language']

X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=False)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=False)

# Alinear columnas entre train y test
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print(f"Dimensiones finales de X_train: {X_train.shape}")
print(f"Dimensiones finales de X_test:  {X_test.shape}")


In [ ]:
print(f"\nColumnas del modelo:")
for col in X_train.columns:
    print(f"  - {col}")
print(f"\nTipos de dato:")
print(X_train.dtypes.value_counts())


## Fase 7: Validación de Supuestos Pre-Modelo
Verificar la distribución del target y si necesitamos normalización.


In [ ]:
# Distribucion del target en train (regresion: revisamos sesgo, no balance de clases)
print("--- DISTRIBUCION DEL TARGET (TRAIN) ---")
print(y_train.describe().round(3))
skew = y_train.skew()
print(f"\nSkewness: {skew:.3f}")
if abs(skew) > 1:
    print("ADVERTENCIA: Target sesgado. Considerar transformar con log o sqrt.")
else:
    print("Distribucion aceptable. No se requiere transformacion del target.")

# Random Forest y XGBoost son basados en arboles -> NO necesitan normalizacion
print(f"\n--- INTEGRIDAD ---")
print(f"NaN en X_train: {X_train.isna().sum().sum()}")
print(f"NaN en X_test:  {X_test.isna().sum().sum()}")
print(f"NaN en y_train: {y_train.isna().sum()}")
print(f"NaN en y_test:  {y_test.isna().sum()}")


## Fase 8: Entrenamiento del Modelo Base (Baseline)
Random Forest Regressor como línea base, luego XGBoost Regressor como modelo avanzado. Como referencia incluimos también el `DummyRegressor` (predice siempre la media), que actúa como "azar" para regresión.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluar(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

# MODELO 0: DUMMY (predice siempre la media del train) - linea de "azar" para regresion
print("=" * 50)
print("MODELO 0: DUMMY (predice la media)")
print("=" * 50)
modelo_dummy = DummyRegressor(strategy='mean')
modelo_dummy.fit(X_train, y_train)
pred_test_dummy = modelo_dummy.predict(X_test)
mae_d, rmse_d, r2_d = evaluar(y_test, pred_test_dummy)
print(f"MAE: {mae_d:.4f} | RMSE: {rmse_d:.4f} | R2: {r2_d:.4f}")

# MODELO 1: RANDOM FOREST REGRESSOR (Baseline)
print("\n" + "=" * 50)
print("MODELO 1: RANDOM FOREST REGRESSOR (Baseline)")
print("=" * 50)

modelo_rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

modelo_rf.fit(X_train, y_train)

pred_train_rf = modelo_rf.predict(X_train)
pred_test_rf = modelo_rf.predict(X_test)

mae_train_rf, rmse_train_rf, r2_train_rf = evaluar(y_train, pred_train_rf)
mae_test_rf, rmse_test_rf, r2_test_rf = evaluar(y_test, pred_test_rf)

print(f"\nTRAIN  - MAE: {mae_train_rf:.4f} | RMSE: {rmse_train_rf:.4f} | R2: {r2_train_rf:.4f}")
print(f"TEST   - MAE: {mae_test_rf:.4f}  | RMSE: {rmse_test_rf:.4f}  | R2: {r2_test_rf:.4f}")
print(f"Gap R2 (Train - Test): {(r2_train_rf - r2_test_rf):.4f}")

if r2_train_rf - r2_test_rf > 0.20:
    print("\nDiagnostico: OVERFITTING (memorizo el pasado)")
elif r2_test_rf < 0.05:
    print("\nDiagnostico: UNDERFITTING (no aprendio mas que predecir la media)")
else:
    print("\nDiagnostico: Modelo razonable")


In [ ]:
import xgboost as xgb

# MODELO 2: XGBOOST REGRESSOR
print("=" * 50)
print("MODELO 2: XGBOOST REGRESSOR")
print("=" * 50)

modelo_xgb = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    objective='reg:squarederror',
)

modelo_xgb.fit(X_train, y_train)

pred_train_xgb = modelo_xgb.predict(X_train)
pred_test_xgb = modelo_xgb.predict(X_test)

mae_train_xgb, rmse_train_xgb, r2_train_xgb = evaluar(y_train, pred_train_xgb)
mae_test_xgb, rmse_test_xgb, r2_test_xgb = evaluar(y_test, pred_test_xgb)

print(f"\nTRAIN  - MAE: {mae_train_xgb:.4f} | RMSE: {rmse_train_xgb:.4f} | R2: {r2_train_xgb:.4f}")
print(f"TEST   - MAE: {mae_test_xgb:.4f}  | RMSE: {rmse_test_xgb:.4f}  | R2: {r2_test_xgb:.4f}")
print(f"Gap R2 (Train - Test): {(r2_train_xgb - r2_test_xgb):.4f}")

if r2_train_xgb - r2_test_xgb > 0.20:
    print("\nDiagnostico: OVERFITTING")
elif r2_test_xgb < 0.05:
    print("\nDiagnostico: UNDERFITTING")
else:
    print("\nDiagnostico: Modelo razonable")



## Fase 9: Evaluación e Interpretación de Resultados
Para regresión: scatter de predicho vs real, histograma de residuos y feature importance del mejor modelo.


In [ ]:
# Elegir el mejor modelo automaticamente (mayor R2 en test)
if r2_test_xgb >= r2_test_rf:
    mejor_nombre = "XGBoost"
    mejor_pred = pred_test_xgb
    mejor_modelo = modelo_xgb
    mae_mejor, rmse_mejor, r2_mejor = mae_test_xgb, rmse_test_xgb, r2_test_xgb
else:
    mejor_nombre = "Random Forest"
    mejor_pred = pred_test_rf
    mejor_modelo = modelo_rf
    mae_mejor, rmse_mejor, r2_mejor = mae_test_rf, rmse_test_rf, r2_test_rf

print(f"Mejor modelo: {mejor_nombre}")
print(f"  MAE:  {mae_mejor:.4f}")
print(f"  RMSE: {rmse_mejor:.4f}")
print(f"  R2:   {r2_mejor:.4f}")


In [ ]:
# Scatter Predicho vs Real + Histograma de Residuos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicho vs Real
axes[0].scatter(y_test, mejor_pred, alpha=0.4, color='steelblue', edgecolor='white')
lo, hi = float(min(y_test.min(), mejor_pred.min())), float(max(y_test.max(), mejor_pred.max()))
axes[0].plot([lo, hi], [lo, hi], 'r--', label='Prediccion perfecta')
axes[0].set_xlabel('Rating real')
axes[0].set_ylabel('Rating predicho')
axes[0].set_title(f'{mejor_nombre} - Predicho vs Real (R2={r2_mejor:.3f})')
axes[0].legend()

# Histograma de residuos
residuos = y_test - mejor_pred
axes[1].hist(residuos, bins=40, color='lightcoral', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_xlabel('Residuo (real - predicho)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title(f'Distribucion de Residuos (MAE={mae_mejor:.3f})')

plt.tight_layout()
plt.show()

print(f"\nResiduos: media={residuos.mean():.4f}, std={residuos.std():.4f}")
print(f"Si la media esta cerca de 0, el modelo no tiene sesgo sistematico.")


In [ ]:
# Tabla comparativa de modelos
comparacion = pd.DataFrame({
    'Modelo':       ['Dummy (media)', 'Random Forest', 'XGBoost'],
    'MAE Test':     [round(mae_d, 4),  round(mae_test_rf, 4),  round(mae_test_xgb, 4)],
    'RMSE Test':    [round(rmse_d, 4), round(rmse_test_rf, 4), round(rmse_test_xgb, 4)],
    'R2 Test':      [round(r2_d, 4),   round(r2_test_rf, 4),   round(r2_test_xgb, 4)],
})
print(f"\n--- COMPARACION DE MODELOS (TEST) ---")
print(comparacion.to_string(index=False))
print(f"\nMejor modelo: {mejor_nombre} (R2={r2_mejor:.4f})")


In [ ]:
importancias = pd.Series(
    mejor_modelo.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importancias.head(20).plot(kind='barh', color='steelblue')
plt.title(f'Importancia de Variables - {mejor_nombre} (Top 20)')
plt.xlabel('Importancia')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nRanking de importancia (Top 20):")
for feat, imp in importancias.head(20).items():
    marker = " <<< ESTRELLA" if imp > 0.10 else ""
    print(f"  {feat}: {imp:.4f}{marker}")

print("\nVariables ignoradas (importancia < 1%):")
ignoradas = importancias[importancias < 0.01].index.tolist()
print(f"  {ignoradas if ignoradas else 'Ninguna'}")


## Fase 10: Iteración y Mejora / Conclusiones

### Lectura del resultado
- **R² > 0**: el modelo aporta valor sobre predecir la media. El R² del Dummy siempre es ~0 en test.
- **MAE bajo en una escala 1-10**: un MAE de 0.5 significa que en promedio el modelo se equivoca medio punto de calificación, lo cual es bastante usable.
- **Gap R² Train/Test pequeño (< 0.20)**: el modelo generaliza bien.
- **Histograma de residuos centrado en 0** sin colas largas: las predicciones no tienen sesgo sistemático.

### Qué aprendimos
1. **Las features categóricas (genre, platform, country, language) son la columna vertebral** una vez removidos los scores derivados.
2. **`votes_per_year` aporta más señal que `votes` crudo** porque descuenta la antigüedad del título.
3. **Eliminar `weighted_rating`, `engagement_score`, `popularity_score` y `trending_score` es no-negociable** — la documentación del dataset las describe como derivadas de rating/votes; usarlas inflaría artificialmente el R².
4. La división temporal por `release_year` permite detectar **concept drift**: si el modelo cae mucho en el futuro, los gustos del público han cambiado.

### Para mejorar más allá del baseline
Se necesitarían datos que este dataset no provee crudos:
- **Embeddings de las descripciones** (ver Fase 10.5 — sentamos la base con TF-IDF).
- **Datos del cast / director** (señal fuerte de calidad esperada).
- **Tiempo de visualización promedio por usuario** (engagement real, no derivado).
- **Presupuesto de producción** (correlaciona con calidad esperada).


---
## Fase 10.5 (Bonus): Sistema de Recomendación Content-Based

El dataset documenta explícitamente como caso de uso "Recommendation Systems (Netflix-style)" e incluye `combined_features` y `description` como campos NLP-ready. Aprovechamos esto para construir un recomendador **content-based** sencillo pero funcional, complementario al modelo de regresión.

**Idea:** Para cada título representamos su descripción (`combined_features`) como vector TF-IDF. Cuando un usuario pide recomendaciones para un título dado, devolvemos los N títulos con **mayor similitud coseno**.

**Por qué encaja con la plataforma Auto Profiling:**
- No requiere `rating` como input → es ortogonal al modelo de regresión.
- Funciona sobre los mismos datos sin entrenamiento supervisado adicional.
- Permite mostrar resultados cualitativos en el dashboard ("Si te gustó X, también te gustarán Y, Z, W").


In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Trabajamos sobre el DataFrame de texto que reservamos en Fase 2.
# El dataset sintetico genera "clones" del mismo concepto: 'Silent Code 146',
# 'Silent Code 2270', 'Silent Code 332' comparten contenido salvo el numero del
# titulo. TF-IDF con min_df=2 descarta esos numeros unicos -> vectores identicos
# -> similitud 1.0. Solucion: deduplicar por el TITULO BASE (sin sufijo numerico)
# para quedarnos con un representante por concepto.
text_df = text_data.dropna(subset=['combined_features']).copy()
text_df['title_base'] = text_df['title'].str.replace(r'\s+\d+$', '', regex=True)
text_df = (text_df
           .drop_duplicates(subset=['title_base', 'type', 'genre', 'platform'])
           .reset_index(drop=True))

# 1. Vectorizar combined_features con TF-IDF
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
)
tfidf_matrix = vectorizer.fit_transform(text_df['combined_features'].astype(str))

print(f"Matriz TF-IDF: {tfidf_matrix.shape}")
print(f"Vocabulario: {len(vectorizer.vocabulary_):,} tokens")
print(f"Titulos unicos tras deduplicar por concepto: {len(text_df):,}")
print(f"Ejemplos de titulos base unicos: {text_df['title_base'].unique()[:8].tolist()}")


In [ ]:
# 2. Calcular similitud coseno y construir indice por titulo
indice_titulos = pd.Series(text_df.index, index=text_df['title']).drop_duplicates()

def recomendar(titulo: str, top_n: int = 5) -> pd.DataFrame:
    """Devuelve los top_n titulos mas similares al titulo dado."""
    if titulo not in indice_titulos:
        coincidencias = text_df[text_df['title'].str.contains(titulo, case=False, na=False)]
        if coincidencias.empty:
            raise ValueError(f"Titulo no encontrado: {titulo}")
        titulo = coincidencias.iloc[0]['title']
        print(f"Coincidencia parcial usada: '{titulo}'")

    idx = indice_titulos[titulo]
    if hasattr(idx, '__iter__'):
        idx = idx.iloc[0] if hasattr(idx, 'iloc') else list(idx)[0]

    sim_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    sim_scores[idx] = -1  # excluir el propio titulo

    top_idx = sim_scores.argsort()[::-1][:top_n]
    resultado = text_df.iloc[top_idx][['title', 'type', 'genre', 'platform']].copy()
    resultado['similarity'] = sim_scores[top_idx].round(4)
    return resultado.reset_index(drop=True)

# 3. Ejemplo: recomendaciones para los 3 primeros titulos del dataset
print("=" * 60)
print("EJEMPLO DE RECOMENDACIONES")
print("=" * 60)

titulos_ejemplo = text_df['title'].head(3).tolist()
ejemplos = []

for t in titulos_ejemplo:
    print(f"\n--- Recomendaciones para: '{t}' ---")
    recomendaciones = recomendar(t, top_n=5)
    print(recomendaciones.to_string(index=False))
    ejemplos.append({"input": t, "recomendaciones": recomendaciones})


## Fase 11: Exportación del Data Contract
Generación automática del JSON estructurado para la plataforma interactiva de Auto Profiling.


In [ ]:
# Instalar helper desde el repositorio
!pip install -q git+https://github.com/Triplerush/auto_profilling.git#subdirectory=helper


In [ ]:
from auto_profiling_export import Report, Section, chart, dataframe

# --- Crear reporte ---
report = Report(
    title="OTT Movies & Series - Analisis",
    description="Prediccion de rating (regresion) + sistema de recomendacion content-based con TF-IDF en plataformas de streaming.",
    author="Triplerush",
    tags=["machine-learning", "ott", "streaming", "regresion", "nlp", "recomendacion", "xgboost"],
    # colab_url="https://colab.research.google.com/drive/xxx",
)

# --- KPIs ---
report.add_kpi("Filas originales", f"{len(data):,}", "Dataset crudo cargado", severity="ok")
report.add_kpi("Filas modelo", f"{len(df_modelo):,}", f"Tras limpieza + Feature Engineering ({len(df_modelo)/len(df)*100:.1f}%)", severity="ok")
report.add_kpi("R2 XGBoost", f"{r2_test_xgb:.3f}", f"Coeficiente de determinacion en test (gap {r2_train_xgb-r2_test_xgb:.3f})", severity="ok")
report.add_kpi("MAE XGBoost", f"{mae_test_xgb:.3f}", "Error absoluto medio en escala 1-10", severity="ok")
report.add_kpi("RMSE XGBoost", f"{rmse_test_xgb:.3f}", "Raiz del error cuadratico medio", severity="ok")
report.add_kpi("Features", f"{X_train.shape[1]}", "Columnas del modelo final tras One-Hot", severity="ok")
report.add_kpi("Vocabulario TF-IDF", f"{len(vectorizer.vocabulary_):,}", "Tokens del recomendador NLP", severity="ok")


In [ ]:
# --- Seccion 1: Ingesta y Limpieza ---
sec1 = Section("Ingesta y Limpieza", f"Dataset original: {len(data):,} filas x {data.shape[1]} columnas.")
sec1.add({
    "type": "metric_grid",
    "title": "Resumen de Limpieza",
    "metrics": [
        {"label": "Filas originales", "value": f"{len(data):,}", "severity": "ok"},
        {"label": "Columnas eliminadas", "value": "10", "severity": "warning"},
        {"label": "Dataset limpio", "value": f"{len(df):,} x {df.shape[1]}", "severity": "ok"},
        {"label": "Duplicados exactos", "value": f"{df.duplicated().sum():,}", "severity": "ok"},
    ]
})
report.add_section(sec1)


In [ ]:
# --- Seccion 2: EDA ---
sec2 = Section("Analisis Exploratorio (EDA)", "Distribucion del rating, plataformas y validacion del Feature Engineering")

# Distribucion del rating (target continuo) -> histograma
hist_rating, bin_edges = np.histogram(df['rating'], bins=20)
labels_rating = [f"{bin_edges[i]:.1f}-{bin_edges[i+1]:.1f}" for i in range(len(hist_rating))]
sec2.add(chart.bar(
    labels=labels_rating,
    datasets=[{"label": "Titulos", "data": hist_rating.tolist()}],
    title="Distribucion del Target (rating)",
    config={"x_label": "Rating (bins)", "y_label": "Frecuencia"},
))

# Distribucion por plataforma
plat_counts = df['platform'].value_counts()
sec2.add(chart.bar(
    labels=plat_counts.index.tolist(),
    datasets=[{"label": "Titulos", "data": plat_counts.values.tolist()}],
    title="Distribucion por Plataforma",
    config={"x_label": "Plataforma", "y_label": "Cantidad"},
))

# Rating promedio por genero
genre_rating = df.groupby('genre')['rating'].mean().sort_values(ascending=False)
sec2.add(chart.bar(
    labels=genre_rating.index.tolist(),
    datasets=[{"label": "Rating promedio", "data": genre_rating.round(3).tolist()}],
    title="Rating promedio por Genero",
    config={"x_label": "Genero", "y_label": "Rating promedio"},
))

# Relacion votes_per_year vs rating
bins = pd.qcut(df_modelo['votes_per_year'], q=5, duplicates='drop')
rating_by_bin = df_modelo.groupby(bins, observed=True)['rating'].agg(['mean', 'count'])
sec2.add(chart.bar(
    labels=[str(b) for b in rating_by_bin.index],
    datasets=[
        {"label": "Rating promedio", "data": rating_by_bin['mean'].round(3).tolist()},
        {"label": "Titulos", "data": rating_by_bin['count'].tolist()},
    ],
    title="Rating promedio segun votes_per_year",
    config={"x_label": "votes_per_year (bins)", "y_label": "Valor"},
))
report.add_section(sec2)


In [ ]:
# --- Seccion 3: Estadisticas de Features ---
sec3 = Section("Estadisticas de Features", "Descriptivos de las variables numericas y derivadas")
feat_cols = ['rating', 'years_since_release', 'duration_log', 'votes_log', 'votes_per_year', 'is_long_content']
desc = df[feat_cols].describe().loc[['count', 'mean', 'std', 'min', 'max']].T.round(4).reset_index()
desc.columns = ['Feature', 'Count', 'Mean', 'Std', 'Min', 'Max']
sec3.add(dataframe(desc, title="Estadisticas descriptivas (incluye target)", config={"sortable": True, "searchable": False}))
report.add_section(sec3)


In [ ]:
# --- Seccion 4: Resultados del Modelo (Regresion) ---
sec4 = Section("Resultados del Modelo", "Comparacion Dummy / Random Forest / XGBoost con split temporal")

# Comparacion de metricas
sec4.add(chart.bar(
    labels=["Dummy (media)", "Random Forest", "XGBoost"],
    datasets=[
        {"label": "MAE Test",  "data": [round(mae_d, 4),  round(mae_test_rf, 4),  round(mae_test_xgb, 4)]},
        {"label": "RMSE Test", "data": [round(rmse_d, 4), round(rmse_test_rf, 4), round(rmse_test_xgb, 4)]},
    ],
    title="Comparacion de Errores (menor = mejor)",
    config={"x_label": "Modelo", "y_label": "Error"},
))

sec4.add(chart.bar(
    labels=["Dummy (media)", "Random Forest", "XGBoost"],
    datasets=[
        {"label": "R2 Test", "data": [round(r2_d, 4), round(r2_test_rf, 4), round(r2_test_xgb, 4)]},
    ],
    title="R2 en Test (mayor = mejor, Dummy es la referencia de azar)",
    config={"x_label": "Modelo", "y_label": "R2"},
))

# Tabla detallada
sec4.add(dataframe(comparacion, title="Tabla comparativa de metricas",
                   config={"sortable": True, "searchable": False}))

# Sample de predicciones (primeras 30 filas del test)
sample_pred = pd.DataFrame({
    'rating_real':     y_test.head(30).round(2).tolist(),
    'rating_predicho': pd.Series(mejor_pred).head(30).round(2).tolist(),
    'residuo':         (y_test.head(30) - pd.Series(mejor_pred, index=y_test.index).head(30)).round(2).tolist(),
}).reset_index(drop=True)
sec4.add(dataframe(sample_pred,
                   title=f"Muestra de predicciones (primeras 30) - {mejor_nombre}",
                   config={"sortable": True, "searchable": False}))

report.add_section(sec4)


In [ ]:
# --- Seccion 5: Feature Importance ---
sec5 = Section("Feature Importance", f"Importancia de cada variable en el modelo {mejor_nombre} final")

imp = importancias[importancias > 0.01]
sec5.add(chart.bar_horizontal(
    labels=imp.index.tolist(),
    data=imp.round(4).tolist(),
    title=f"Feature Importance - {mejor_nombre}",
    config={"x_label": "Importancia"},
))
report.add_section(sec5)


In [ ]:
# --- Seccion 6: Sistema de Recomendacion (Bonus NLP) ---
sec6 = Section("Sistema de Recomendacion Content-Based",
               "TF-IDF + similitud coseno sobre 'combined_features'. Dado un titulo, devuelve los N mas parecidos.")

sec6.add({"type": "text", "content": f"""**Como funciona:**

- Se vectorizan las descripciones (`combined_features`) usando **TF-IDF** con n-gramas de 1 y 2 palabras.
- Vocabulario aprendido: **{len(vectorizer.vocabulary_):,} tokens**.
- Similitud entre titulos: **distancia coseno** entre vectores TF-IDF.
- No necesita `rating` como input, asi que es complementario al modelo de regresion."""})

# Mostrar 3 ejemplos como tablas
for ej in ejemplos:
    sec6.add({"type": "text", "content": f"### Recomendaciones para: *{ej['input']}*"})
    sec6.add(dataframe(
        ej['recomendaciones'],
        title=f"Top 5 similares a '{ej['input']}'",
        config={"sortable": True, "searchable": False}
    ))

report.add_section(sec6)


In [ ]:
# --- Seccion 7: Conclusiones ---
sec7 = Section("Conclusiones")
sec7.add({"type": "text", "content": f"""## Hallazgos principales

1. **Regresion sobre `rating` con R2 de {r2_test_xgb:.3f} en test** y MAE de {mae_test_xgb:.3f} puntos en escala 1-10. El modelo no memoriza (gap train/test bajo).
2. **Eliminar `weighted_rating`, `engagement_score`, `popularity_score` y `trending_score` fue critico**: la documentacion las describe como derivadas de rating/votes y son fuga de datos.
3. **`votes_per_year` y los dummies de genero/plataforma** dominan la importancia. La popularidad anualizada descuenta el sesgo de antiguedad.
4. **El recomendador TF-IDF** complementa el modelo: el regresor estima *que tan bueno* sera un titulo; el recomendador encuentra titulos *similares en contenido* a uno que ya gusto.

## Proximos pasos

- Reemplazar TF-IDF por **embeddings densos** (sentence-transformers) para el recomendador.
- Combinar la prediccion del regresor con la similitud del recomendador en un *re-ranker* hibrido.
- Anadir datos externos del cast/director y entrenar un modelo calibrado para producir probabilidades de "hit"."""})
report.add_section(sec7)


In [ ]:
# --- Exportar modelo de regresion ---
# Nota: el helper actual no tiene un parametro `task`; el model-service
# solo expone .predict() y deduce task por la presencia de predict_proba.
# Para un regresor, el endpoint /predict devolvera {"prediction": <float>,
# "probability": null, "confidence": null}, lo cual es correcto para regresion.
from auto_profiling_export import export_model

model_info = export_model(
    model=mejor_modelo,
    analysis_id="ott-movies-series",
    input_schema=X_train.columns.tolist(),
    sample_input={col: float(X_test.iloc[0][col]) for col in X_train.columns},
    metrics={
        "mae":  round(mae_mejor, 4),
        "rmse": round(rmse_mejor, 4),
        "r2":   round(r2_mejor, 4),
    },
    output_dir=".",
)
report.set_model(model_info)
print(f"Modelo exportado: {model_info['artifact']}")
print(f"Features: {model_info['input_schema']}")
print(f"Metricas: {model_info['metrics']}")


In [ ]:
# --- Adjuntar catalogo de items para el endpoint /recommend ---
# El platform-frontend usara este catalogo para mostrar el Top 5
# de titulos cuyo rating real este mas cerca del rating predicho
# y que coincidan en mas categorias con la entrada del usuario.
report.set_catalog(
    df=data,
    display_fields=['title', 'type', 'genre', 'platform', 'country', 'language', 'release_year', 'duration_minutes', 'rating', 'votes'],
    match_fields=['type', 'genre', 'platform', 'country', 'language'],
    rating_field='rating',
    top_k=5,
    label='Top 5 recomendaciones',
)
print(f"Catalogo adjunto: {len(data):,} items")


In [ ]:
# --- Exportar Data Contract ---
report.save("ott-movies-series.json")
print(f"Data Contract exportado: ott-movies-series.json")
